## Source

Estimates are derived from the Global Forest Watch website (conversion emissions) for Uganda by combining estimates of total above-ground biomass C stocks in forests with total areas to generate an estimate of a per-unit factor to directly estimate conversion emissions, then comparing this with conversion emissions accounted for in the inventories. 2015 Shows _negative_ conversion emissions from converting forests to cropland (?), so its value is instead determined as a mean of values from 2016-2019.


In [9]:
import numpy as np
import pandas as pd
import os
import pathlib

import warnings
warnings.filterwarnings("ignore")

import sys

dir_git = "/Users/usuario/git/sisepuede"
if dir_git not in sys.path:
    sys.path.append(dir_git)
    
import sisepuede.utilities._toolbox as sf
import utils.common_data_needs as cdn


Detected IPython. Loading juliacall extension. See https://juliapy.github.io/PythonCall.jl/stable/compat/#IPython


Precompiling NemoMod...
Info Given NemoMod was explicitly requested, output will be shown live 
ERROR: Method overwriting is not permitted during Module precompilation. Use `__precompile__(false)` to opt-out of precompilation.
   1585.2 ms  ? NemoMod
[ Info: Precompiling NemoMod [a3c327a0-d2f0-11e8-37fd-d12fd35c3c72] 
ERROR: Method overwriting is not permitted during Module precompilation. Use `__precompile__(false)` to opt-out of precompilation.
┌ Info: Skipping precompilation due to precompilable error. Importing NemoMod [a3c327a0-d2f0-11e8-37fd-d12fd35c3c72].
└   exception = Error when precompiling module, potentially caused by a __precompile__(false) declaration in the module.


In [3]:
%load_ext autoreload
%autoreload 2

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [11]:

# get SISEPUEDE elements
dict_ssp = cdn._setup_sisepuede_elements()
examples = cdn._SISEPUEDE_EXAMPLES
struct = cdn._SISEPUEDE_FILE_STRUCTURE

# shortcuts
matt = dict_ssp.get("model_attributes", )
models = dict_ssp.get("models", )
regions = dict_ssp.get("regions", )
time_periods = dict_ssp.get("time_periods", )

# setup region
_REGION_NAME = "uganda"
_REGION_ISO = regions.return_region_or_iso(_REGION_NAME, return_type = "iso")


In [16]:
# Set up paths
CURRENT_DIR_PATH = pathlib.Path(os.path.abspath("."))
OUTPUT_DATA_DIR_PATH = CURRENT_DIR_PATH.joinpath("output_data")
INPUT_DATA_DIR_PATH = CURRENT_DIR_PATH.joinpath("input_data")

In [13]:

modvar_frst_frac = matt.get_variable(models.model_enercons.modvar_frst_conversion_frac_for_fuelwood, )
field_frst_frac = modvar_frst_frac.fields[0]

In [39]:

path_file = INPUT_DATA_DIR_PATH.joinpath("estimating_fraction_of_conversion_available_for_biomass.xlsx")
df = pd.read_excel(
    path_file,
    skiprows = 3,
)

field_frac = "est frac available for bmass"
field_yr = "loss by year"
df = (
    df
    .get([field_yr, field_frac])
    .rename(
        columns = {
            field_yr: time_periods.field_year,
            field_frac: field_frst_frac,
        }
    )
)

val_mean = np.mean(
    df[
        df[time_periods.field_year] >= 2015
    ][field_frst_frac]
    .dropna(), 
)
df[field_frst_frac] = (
    df[field_frst_frac]
    .fillna(
        val_mean,
    )
)

# build with an assumption that removals will continue as a same share until 2035,
# then decline to 2070 as biomass as a key driver of deforestation decreases (in line with all docs)
df_frac_1 = pd.DataFrame(
    {
        time_periods.field_year: range(2015, 2036)
    }
)
df_frac_1 = (
    pd.merge(
        df_frac_1,
        df,
        how = "left",
    )
    .fillna(val_mean)
)


df_frac_2 = pd.DataFrame(
    {
        time_periods.field_year: range(2015, 2100)
    }
)
df_frac_2 = (
    pd.merge(
        df_frac_2,
        pd.concat(
            [
                df_frac_1,
                pd.DataFrame(
                    {
                        time_periods.field_year: [2050, 2070],
                        field_frst_frac: [0.5, 0.25]
                    }
                )
            ]
        ),
        how = "left",
    )
    .interpolate()
    .ffill()
)

In [44]:
cdn.file_name_from_variable(modvar_frst_frac, )

'FRACTION_OF_FOREST_CONVERSIONS_AVAILABLE_FOR_FUELWOOD.csv'

In [47]:
fn_write = cdn.file_name_from_variable(modvar_frst_frac, )
sf._write_csv(
    df_frac_2,
    cdn._PATH_OUTPUTS.joinpath(fn_write),
)


True